In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
import requests
from selenium.webdriver.common.by import By

In [2]:
from time import sleep

def find_coords_from_maps_url(url:str)->tuple:
    fsplit = url.split('!3d')[1].split('!4d')
    return (fsplit[0], fsplit[1].split('!16s')[0])

def find_coords_from_address(address:str, sleep_timer:float=5,
                             max_attempts:int=3)->tuple:
    attempt = 0
    with webdriver.Chrome() as browser:
        browser.get(f"https://www.google.com/maps/place/{address.replace(' ', '+')}")
        sleep(sleep_timer)
        url = browser.current_url

        try:
            coords = find_coords_from_maps_url(url=url)
        except:
            attempt += 1
            coords = find_coords_from_address(address=address) \
                        if max_attempts > attempt else None

        return coords
    

In [3]:
address="1008 S 4th St, Champaign, IL"



coordinates

('40.1061258', '-88.2329753')

In [4]:

def get_places_from_address(address:str, query:str) -> dict:
    coordinates = find_coords_from_address(address=address, sleep_timer=4, max_attempts=4)
    
    zoom_level = 15

    times_scrolled = 5
    scroll_amount = 1000

    url = f'https://www.google.com/maps/search/{query}/@{coordinates[0]},{coordinates[1]},{zoom_level}z'

    with webdriver.Chrome() as browser:
        browser.set_window_size(1024, 768)
        browser.get(url)

        scroll_origin = ScrollOrigin.from_viewport(150, 200)

        for i in range(times_scrolled):
            ActionChains(browser)\
                .scroll_from_origin(scroll_origin, 0, scroll_amount)\
                .perform()
            sleep(2)
        #browser.execute_script("window.scrollBy(200,3000)","")

        soup = BeautifulSoup(browser.page_source, 'html.parser')

        

        results_div = soup.find('div', {'aria-label': f'Results for {query}'})

        
        divs = {div.find('div').find('a').attrs['aria-label']: div.find('div').find('a').attrs['href'] for div in results_div.find_all('div') if str(div)[:5] == '<div>' and div.find('div').find('a')}
        whole_divs = [div for div in results_div.find_all('div') if str(div)[:5] == '<div>' and div.find('div').find('a')]


        data = {}

        for name, href in divs.items():
            data[name] = find_coords_from_maps_url(href)
    return data



In [40]:
import json
json.dump(data, open('res.json', 'w'))

In [6]:
whole_divs[1]

<div><div class="Nv2PK THOPZb CpccDe" jsaction="mouseover:pane.wfvdle11;mouseout:pane.wfvdle11"><a aria-label="Four American Grill - Four Breakfast &amp; More" class="hfpxzc" href="https://www.google.com/maps/place/Four+American+Grill+-+Four+Breakfast+%26+More/data=!4m7!3m6!1s0x880cd7d73e2683eb:0x2b44c8a5b3a0c92!8m2!3d40.1101023!4d-88.2333023!16s%2Fg%2F11ght3fch3!19sChIJ64MmPtfXDIgRkgw6W4pMtAI?authuser=0&amp;hl=en&amp;rclk=1" jsaction="pane.wfvdle11;focus:pane.wfvdle11;blur:pane.wfvdle11;auxclick:pane.wfvdle11;keydown:pane.wfvdle11;clickmod:pane.wfvdle11" jslog="12690;track:click,contextmenu;mutable:true;metadata:WyIwYWhVS0V3aUxnc3lqLUllRUF4WGJKVFFJSGIxZUJMd1E4QmNJTXlnQSIsbnVsbCwyXQ=="></a><div class="rWbY0d"></div><div class="bfdHYd Ppzolf OFBs3e"><div class="rgFiGf OyjIsf"></div><div class="hHbUWd"></div><div class="rSy5If"></div><div class="lI9IFe"><div class="y7PRA"><div class="Lui3Od T7Wufd"><div class="Z8fK3b"><div class="OyjIsf"></div><div class="UaQhfb fontBodyMedium"><div clas

In [10]:
## TRIED TO HAVE AN AI INSPECT THE CODE AND RETRIEVE DATA INSTEAD OF HARDCODING
## FOUND IT TOO UNRELIABLE WITH LOCAL AIS
## CHATGPT 3 WAS GOOD BUT WOULDVE BEEN $1 PER REQUEST, SINCE THE HTML WAS SUPER LONG

import json
from IPython.display import clear_output

coordinates

url = 'http://localhost:11434/api/generate'

headers = {'Content-Type':'application/json'}

location_data = []

code = str(whole_divs[3])

data = {
    "model": "maps-scrapper",
    "prompt": f'''{code}''',
    "stream": True
}

r_str = ''

s = requests.Session()
with s.post(url, headers=headers, data=json.dumps(data), stream=True) as r:
    for line in r.iter_lines():
        r_str += json.loads(line.decode('utf-8'))['response']
        clear_output(wait=True)
        print(r_str)
        print(location_data)

location = r_str.split('```')[1].split('```')[0]
print(location)
location_data += [json.loads(location)]
    




 










































































































































































































































































































[]


KeyboardInterrupt: 

In [2]:
whole_divs[0]

NameError: name 'whole_divs' is not defined

{'State and Lake Chicago Tavern': ('41.885946', '-87.627687'), 'Taco Maya': ('41.8711881', '-87.6273082'), 'The Gage': ('41.8813268', '-87.6246925'), 'Staytion Market & Bar': ('41.8866709', '-87.6282392'), 'Elephant & Castle': ('41.8793112', '-87.6314087'), 'Oriole': ('41.8861692', '-87.6450939'), 'First Draft': ('41.8734567', '-87.6305'), "Hoyt's Chicago": ('41.8873941', '-87.625694'), "Jimmy's Gyros & Grill": ('41.8796007', '-87.6267684'), 'Taco Bell Cantina': ('41.8853812', '-87.6265345'), 'Jubilee Juice & Grill': ('41.8840501', '-87.6476932'), 'Seven Bar and Restaurant': ('41.8846364', '-87.6164952'), 'Plymouth Restaurant & Rooftop Bar': ('41.8774756', '-87.6285532'), 'Hoots South Loop': ('41.8661586', '-87.6403262'), "Rafael's Restaurant": ('41.8754306', '-87.6245382'), 'Nia Restaurant': ('41.884169', '-87.647645'), 'Havana Grill': ('41.889655', '-87.6312732'), 'COLUMBUS TAP': ('41.8861637', '-87.621083'), 'Roanoke Restaurant': ('41.8818554', '-87.6319806'), 'Burrito Beach': ('41.

('40.1061258', '-88.2329753')